# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AxelYoel/FlyRank-AI-Internship---Axel-Yoel-Chandra/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
%pip -q install duckdb
import duckdb, os
from google.colab import userdata

hf_token = userdata.get("HF_token")
os.environ["HF_token"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [2]:
df = con.sql(f"""
    WITH daily AS (
        SELECT content_hash_id, gsc_avg_position, gsc_impressions, gsc_clicks
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available = TRUE AND gsc_avg_position > 0 AND gsc_impressions > 0
    ),
    rollup AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_avg_position * gsc_impressions) / SUM(gsc_impressions) AS weighted_avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 50
    ),
    tiered AS (
        SELECT *,
            CASE
                WHEN weighted_avg_position <= 3 THEN 'tier_1_1-3'
                WHEN weighted_avg_position <= 10 THEN 'tier_2_4-10'
                WHEN weighted_avg_position <= 20 THEN 'tier_3_11-20'
                ELSE 'tier_4_21plus'
            END AS position_tier
        FROM rollup
    ),
    benchmark(position_tier, expected_ctr) AS (
        VALUES
            ('tier_1_1-3', 0.002811), ('tier_2_4-10', 0.002353),
            ('tier_3_11-20', 0.002226), ('tier_4_21plus', 0.000748)
    ),
    labeled AS (
        SELECT
            t.content_hash_id, t.impressions, t.clicks, t.weighted_avg_position,
            b.expected_ctr,
            (b.expected_ctr * t.impressions) - t.clicks AS lost_clicks
        FROM tiered t
        JOIN benchmark b USING (position_tier)
    )
    SELECT
        l.content_hash_id, l.lost_clicks, l.impressions, l.clicks,
        l.weighted_avg_position, l.expected_ctr,
        c.search_volume, c.competition_level, c.content_type,
        CASE WHEN c.content_updated_date <= DATE '2026-03-31'
             THEN DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')
             ELSE NULL END AS days_since_update_capped,
        c.word_count
    FROM labeled l
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON l.content_hash_id = c.content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
df.columns.tolist()

['content_hash_id',
 'lost_clicks',
 'impressions',
 'clicks',
 'weighted_avg_position',
 'expected_ctr',
 'search_volume',
 'competition_level',
 'content_type',
 'days_since_update_capped',
 'word_count']

In [4]:
df['search_volume'].describe()

,search_volume
count,111905.0
mean,146.078281
std,2047.170943
min,0.0
25%,0.0
50%,10.0
75%,30.0
max,246000.0


In [5]:
df['search_volume'].quantile([0.1,0.25,0.5,0.75,0.9])

,search_volume
0.10,0
0.25,0
0.50,10
0.75,30
0.90,110


In [6]:
df['search_volume'].isna().sum()

np.int64(2542)

In [7]:
df.groupby('content_type')['search_volume'].apply(lambda s: (s==0).mean())

,search_volume
content_type,
comparison article,0.999398
feedly article,<NA>
keyword article,0.37251


In [11]:
df['content_type'].value_counts(dropna=False)

,count
content_type,
keyword article,111197
comparison article,1661
feedly article,1589


In [12]:
df[df['content_type'] == 'feedly article']['search_volume'].isna().sum()

np.int64(1589)

In [13]:
df[df['content_type'] == 'feedly article']['search_volume'].value_counts(dropna=False)

,count
search_volume,
<NA>,1589


In [14]:
df[df['content_type'] == 'feedly article'][['word_count', 'competition_level', 'days_since_update_capped']].isna().mean()

,0
word_count,0.027061
competition_level,1.000000
days_since_update_capped,0.589050


In [16]:
df[df['content_type'] == 'comparison article']['search_volume'].isna().sum()

np.int64(0)

In [17]:
df[df['content_type'] == 'comparison article']['search_volume'].value_counts(dropna=False)

,count
search_volume,
0,1660
10,1


In [18]:
df[df['content_type'] == 'comparison article'][['word_count', 'competition_level', 'days_since_update_capped']].isna().mean()

,0
word_count,0.000000
competition_level,0.000000
days_since_update_capped,0.910295


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.